In [114]:
import numpy as np
import pandas as pd

import NeuralNet
import layer
import dataset

In [115]:
data = dataset.load()
data.head()

,Diagnosis,Mean Radius,Mean Texture,Mean Perimeter,Mean Area,Mean Smoothness,Mean Compactness,Mean Concavity,Mean Concave Points,Mean Symmetry,...,Worst Radius,Worst Texture,Worst Perimeter,Worst Area,Worst Smoothness,Worst Compactness,Worst Concavity,Worst Concave Points,Worst Symmetry,Worst Fractal Dimension
0,0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,0,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,0,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,0,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,0,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [116]:
def kfolds(data, nfolds):
    folds = []
    foldLength = int(data.shape[0]/nfolds)

    data = data.reindex(np.random.permutation(data.index))                                                          
    data = data.reset_index(drop = True)
    
    idx = 0
    lidx = foldLength - 1
    for i in range(nfolds):
        folds.append(data.loc[idx : lidx])
        idx += foldLength
        lidx += foldLength
    return folds

In [117]:
def kfoldsplit(folds):
    train_test = []

    for i in range(len(folds)):
        temp = []
        folds_copy = folds.copy()

        test = folds_copy.pop(i)

        temp.append(pd.concat(folds_copy))
        temp.append(test)
        
        train_test.append(temp)
    return train_test

In [118]:
def kfoldcv(ann, data, activations = [2, 2], nodes = [2, 2], nfolds = 5):
    folds = kfolds(data, nfolds)
    train_test = kfoldsplit(folds)

    accuracies = []
    fold = 1
    for i in train_test:
        print("\nCross Validating Fold", fold)
        X_test = np.asarray(i[-1].drop(i[0].columns[0], axis = 1)).T
        y_test = np.array(i[-1].iloc[:,0]).T

        X_test = dataset.normalize(X_test)

        X_train = np.array(i[0].drop(i[0].columns[0], axis = 1)).T
        y_train = np.array(i[0].iloc[:,0]).T
        
        X_train = dataset.normalize(X_train)

        ann.input = X_train
        ann.output = y_train

        ann.setLayers(activations, nodes)
        ann.get_properties()

        print("\nTraining Neural Network..")
        avgloss, accuracy, time = ann.train_sgd()

        print("\nAverage loss over", ann.training_epochs, "epoch(s) in fold", fold, "is", avgloss)
        print("Training accuracy over", ann.training_epochs, "epoch(s) in fold", fold, "is", round(accuracy * 100, 2), "%")
        print("Training time over", ann.training_epochs, "epoch(s) in fold", fold, "is", time, "seconds")

        print("\nTesting Neural Network..")
        accuracy = ann.test(X_test, y_test)
        accuracies.append(accuracy)
        print("Testing accuracy for fold", fold, "is", round(accuracy * 100, 2), "%")

        # reset ann
        ann = NeuralNet.ANN([0], [0])
        fold += 1
    
    print("\nThe testing accuracies are:", ", ".join(str(i) for i in accuracies))
    print("Average accuracy is", round((sum(accuracies)/nfolds) * 100, 2), "%")
    return accuracies

ann = NeuralNet.ANN([0], [0])
kfoldcv(ann, data)


Cross Validating Fold 1
Number of layers: 3
Learning rate: 0.1
Training epochs: 100
Loss function: <function hinge_loss at 0x000001B9B4B0EE60>
Decay: None

Training Neural Network..

Average loss over 100 epoch(s) in fold 1 is 0.022566371681415884
Training accuracy over 100 epoch(s) in fold 1 is 99.34 %
Training time over 100 epoch(s) in fold 1 is 1.4349462985992432 seconds

Testing Neural Network..
Testing accuracy for fold 1 is 96.46 %

Cross Validating Fold 2
Number of layers: 3
Learning rate: 0.1
Training epochs: 100
Loss function: <function hinge_loss at 0x000001B9B4B0EE60>
Decay: None

Training Neural Network..

Average loss over 100 epoch(s) in fold 2 is 0.03393805309734512
Training accuracy over 100 epoch(s) in fold 2 is 99.12 %
Training time over 100 epoch(s) in fold 2 is 1.3191654682159424 seconds

Testing Neural Network..
Testing accuracy for fold 2 is 95.58 %

Cross Validating Fold 3
Number of layers: 3
Learning rate: 0.1
Training epochs: 100
Loss function: <function hinge

[0.9646017699115044,
 0.9557522123893806,
 0.9734513274336283,
 0.9026548672566371,
 0.9469026548672567]